# Finance Data Platform - Environment Deployment

## Executive Summary

This notebook automates the deployment of a production-ready Unity Catalog environment for the Finance Data Platform following enterprise data engineering best practices and the medallion architecture pattern.

### Deployment Scope

**Catalogs:** 3 environment-specific Unity Catalogs  
**Schemas:** 5 schemas per catalog (15 total)  
**Architecture:** Medallion (Bronze-Silver-Gold) + Operational layers  
**Compliance:** Enterprise security and governance standards

---

### Environment Strategy

| Environment | Catalog | Purpose | Retention | Owner |
|-------------|---------|---------|-----------|-------|
| **Development** | `Dev_Finance` | Engineering sandbox and testing | 30 days | Data Engineering |
| **QA** | `QA_Finance` | Integration testing and validation | 90 days | QA Team |
| **Production** | `Prod_Finance` | Live operations and analytics | 7 years | Platform Team |

---

### Data Architecture - Medallion Pattern

```
┌─────────────────────┐
│   SOURCE SYSTEMS    │
│  (ERP, CRM, APIs)  │
└────────┬───────────┘
         │
         ↓ Ingestion
         │
┌────────┴───────────┐
│  BRONZE (Raw Data)  │  ← Append-only, immutable
│  - Audit trail      │
│  - No transformation│
└────────┬───────────┘
         │
         ↓ Cleansing
         │
┌────────┴───────────┐
│  SILVER (Cleansed) │  ← Business rules applied
│  - Validated        │
│  - Conformed        │
└────────┬───────────┘
         │
         ↓ Aggregation
         │
┌────────┴───────────┐
│  GOLD (Metrics)    │  ← Business KPIs
│  - Aggregated       │
│  - BI-optimized     │
└────────┬───────────┘
         │
         ↓
┌────────┴───────────┐
│  BI / ANALYTICS    │
│  (Dashboards, ML)  │
└─────────────────────┘
```

#### Additional Schemas

* **Staging**: Temporary workspace for ETL processing and transformations
* **Archive**: Long-term historical data retention for compliance (7+ years)

---

### Code Standards

This notebook implements enterprise-grade Python standards:

* ✓ Type hints on all functions
* ✓ Comprehensive docstrings (Google style)
* ✓ Class-based architecture for reusability
* ✓ Structured logging (Python logging module)
* ✓ Exception handling with context
* ✓ Configuration management
* ✓ Validation and verification
* ✓ Deployment reporting

---

In [0]:
"""Configuration module for Finance Data Platform environment setup.

This module defines the core configuration for creating Unity Catalog
environments following the medallion architecture pattern.
"""

import logging
from typing import Dict, List, Any
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================================================
# PROJECT CONFIGURATION
# ============================================================================

PROJECT_NAME: str = "Finance"
PROJECT_VERSION: str = "1.0.0"
CREATED_BY: str = "Data Platform Team"
CREATION_DATE: str = datetime.now().strftime("%Y-%m-%d")

# ============================================================================
# ENVIRONMENT DEFINITIONS
# ============================================================================

class EnvironmentConfig:
    """Environment configuration constants."""
    DEV: str = "Dev"
    QA: str = "QA"
    PROD: str = "Prod"
    
    @classmethod
    def get_all_environments(cls) -> List[str]:
        """Return list of all environment names."""
        return [cls.DEV, cls.QA, cls.PROD]

ENVIRONMENTS: List[str] = EnvironmentConfig.get_all_environments()

# ============================================================================
# DATA LAYER DEFINITIONS (MEDALLION ARCHITECTURE)
# ============================================================================

class DataLayerConfig:
    """Data layer configuration for medallion architecture."""
    BRONZE: str = "bronze"
    SILVER: str = "silver"
    GOLD: str = "gold"
    STAGING: str = "staging"
    ARCHIVE: str = "archive"
    
    @classmethod
    def get_medallion_layers(cls) -> List[str]:
        """Return core medallion architecture layers."""
        return [cls.BRONZE, cls.SILVER, cls.GOLD]
    
    @classmethod
    def get_operational_schemas(cls) -> List[str]:
        """Return operational/support schemas."""
        return [cls.STAGING, cls.ARCHIVE]

DATA_LAYERS: List[str] = DataLayerConfig.get_medallion_layers()
OPERATIONAL_SCHEMAS: List[str] = DataLayerConfig.get_operational_schemas()

# ============================================================================
# CATALOG METADATA CONFIGURATION
# ============================================================================

CATALOG_METADATA: Dict[str, Dict[str, str]] = {
    EnvironmentConfig.DEV: {
        "comment": "Development environment for Finance data engineering and testing",
        "owner": "data_engineering_team",
        "purpose": "Development and experimentation",
        "data_retention_days": "30"
    },
    EnvironmentConfig.QA: {
        "comment": "Quality Assurance environment for Finance data validation and testing",
        "owner": "qa_team",
        "purpose": "Testing and validation",
        "data_retention_days": "90"
    },
    EnvironmentConfig.PROD: {
        "comment": "Production environment for Finance business operations and analytics",
        "owner": "data_platform_team",
        "purpose": "Production operations",
        "data_retention_days": "2555"  # 7 years for compliance
    }
}

# ============================================================================
# SCHEMA METADATA CONFIGURATION
# ============================================================================

SCHEMA_METADATA: Dict[str, str] = {
    DataLayerConfig.BRONZE: "Raw data ingestion layer - source system data as-is",
    DataLayerConfig.SILVER: "Cleansed and validated data layer - business rules applied",
    DataLayerConfig.GOLD: "Analytics-ready aggregated data layer - business metrics and KPIs",
    DataLayerConfig.STAGING: "Temporary workspace for data transformations and processing",
    DataLayerConfig.ARCHIVE: "Historical data retention and compliance storage"
}

# ============================================================================
# VALIDATION
# ============================================================================

def validate_configuration() -> bool:
    """Validate configuration integrity.
    
    Returns:
        bool: True if configuration is valid, raises exception otherwise
    """
    try:
        assert len(ENVIRONMENTS) > 0, "No environments defined"
        assert len(DATA_LAYERS) > 0, "No data layers defined"
        assert all(env in CATALOG_METADATA for env in ENVIRONMENTS), \
            "Missing catalog metadata for some environments"
        logger.info("Configuration validation passed")
        return True
    except AssertionError as e:
        logger.error(f"Configuration validation failed: {e}")
        raise

# Run validation
validate_configuration()

# ============================================================================
# DISPLAY CONFIGURATION SUMMARY
# ============================================================================

logger.info(f"Configuration loaded for {PROJECT_NAME} project v{PROJECT_VERSION}")
logger.info(f"Environments: {', '.join(ENVIRONMENTS)}")
logger.info(f"Data Layers: {', '.join(DATA_LAYERS)}")
logger.info(f"Operational Schemas: {', '.join(OPERATIONAL_SCHEMAS)}")
logger.info(f"Created by: {CREATED_BY} on {CREATION_DATE}")

In [0]:
"""Unity Catalog management utilities.

Provides functions for creating and managing Unity Catalog objects
including catalogs, schemas, and related metadata operations.
"""

import logging
from typing import Optional, Dict, Any, Tuple
from pyspark.sql import SparkSession

logger = logging.getLogger(__name__)

# ============================================================================
# CATALOG MANAGEMENT
# ============================================================================

class CatalogManager:
    """Manages Unity Catalog creation and configuration."""
    
    def __init__(self, spark_session: SparkSession):
        """
        Initialize CatalogManager.
        
        Args:
            spark_session: Active Spark session
        """
        self.spark = spark_session
        logger.info("CatalogManager initialized")
    
    def catalog_exists(self, catalog_name: str) -> bool:
        """
        Check if a catalog exists.
        
        Args:
            catalog_name: Name of the catalog to check
            
        Returns:
            bool: True if catalog exists, False otherwise
        """
        try:
            existing_catalogs = self.spark.sql("SHOW CATALOGS").collect()
            return any(row.catalog == catalog_name for row in existing_catalogs)
        except Exception as e:
            logger.error(f"Error checking catalog existence: {e}")
            return False
    
    def create_catalog(
        self, 
        catalog_name: str, 
        comment: str, 
        owner: Optional[str] = None,
        properties: Optional[Dict[str, str]] = None
    ) -> Tuple[bool, str]:
        """
        Create a Unity Catalog if it doesn't exist.
        
        Args:
            catalog_name: Name of the catalog to create
            comment: Description of the catalog
            owner: Optional owner for the catalog
            properties: Optional catalog properties
            
        Returns:
            Tuple[bool, str]: (success_flag, message)
        """
        try:
            # Validate catalog name
            if not self._is_valid_name(catalog_name):
                msg = f"Invalid catalog name: {catalog_name}"
                logger.error(msg)
                return False, msg
            
            # Check if catalog already exists
            if self.catalog_exists(catalog_name):
                msg = f"Catalog '{catalog_name}' already exists - skipping creation"
                logger.info(msg)
                print(f"  ℹ {msg}")
                return False, msg
            
            # Build CREATE CATALOG statement
            create_sql = f"CREATE CATALOG IF NOT EXISTS `{catalog_name}` COMMENT '{comment}'"
            
            # Execute creation
            self.spark.sql(create_sql)
            msg = f"Successfully created catalog: {catalog_name}"
            logger.info(msg)
            print(f"  ✓ Created catalog: {catalog_name}")
            
            return True, msg
            
        except Exception as e:
            msg = f"Error creating catalog '{catalog_name}': {str(e)}"
            logger.error(msg, exc_info=True)
            print(f"  ✗ {msg}")
            return False, msg
    
    def _is_valid_name(self, name: str) -> bool:
        """
        Validate catalog/schema name according to Unity Catalog rules.
        
        Args:
            name: Name to validate
            
        Returns:
            bool: True if valid, False otherwise
        """
        if not name or len(name) == 0:
            return False
        if len(name) > 255:
            return False
        # Basic validation - can be extended
        return True

# ============================================================================
# SCHEMA MANAGEMENT
# ============================================================================

class SchemaManager:
    """Manages Unity Catalog schema creation and configuration."""
    
    def __init__(self, spark_session: SparkSession):
        """
        Initialize SchemaManager.
        
        Args:
            spark_session: Active Spark session
        """
        self.spark = spark_session
        logger.info("SchemaManager initialized")
    
    def schema_exists(self, catalog_name: str, schema_name: str) -> bool:
        """
        Check if a schema exists in a catalog.
        
        Args:
            catalog_name: Parent catalog name
            schema_name: Schema name to check
            
        Returns:
            bool: True if schema exists, False otherwise
        """
        try:
            existing_schemas = self.spark.sql(
                f"SHOW SCHEMAS IN `{catalog_name}`"
            ).collect()
            return any(row.databaseName == schema_name for row in existing_schemas)
        except Exception as e:
            logger.error(f"Error checking schema existence: {e}")
            return False
    
    def create_schema(
        self,
        catalog_name: str,
        schema_name: str,
        comment: str,
        properties: Optional[Dict[str, str]] = None
    ) -> Tuple[bool, str]:
        """
        Create a schema within a catalog if it doesn't exist.
        
        Args:
            catalog_name: Parent catalog name
            schema_name: Schema name to create
            comment: Description of the schema
            properties: Optional schema properties
            
        Returns:
            Tuple[bool, str]: (success_flag, message)
        """
        try:
            # Validate inputs
            if not catalog_name or not schema_name:
                msg = "Catalog name and schema name are required"
                logger.error(msg)
                return False, msg
            
            # Check if schema already exists
            if self.schema_exists(catalog_name, schema_name):
                msg = f"Schema '{catalog_name}.{schema_name}' already exists - skipping"
                logger.info(msg)
                print(f"    ℹ {msg}")
                return False, msg
            
            # Build CREATE SCHEMA statement
            create_sql = f"""
            CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}` 
            COMMENT '{comment}'
            """
            
            # Execute creation
            self.spark.sql(create_sql)
            msg = f"Successfully created schema: {catalog_name}.{schema_name}"
            logger.info(msg)
            print(f"    ✓ Created schema: {catalog_name}.{schema_name}")
            
            return True, msg
            
        except Exception as e:
            msg = f"Error creating schema '{catalog_name}.{schema_name}': {str(e)}"
            logger.error(msg, exc_info=True)
            print(f"    ✗ {msg}")
            return False, msg

# ============================================================================
# INITIALIZE MANAGERS
# ============================================================================

catalog_manager = CatalogManager(spark)
schema_manager = SchemaManager(spark)

logger.info("Unity Catalog management utilities loaded successfully")
print("✓ Catalog and Schema managers initialized")

## Step 1: Create Unity Catalog Catalogs

Creating three environment-specific catalogs for the Finance project.

In [0]:
"""Execute catalog creation for all environments."""

import logging
from typing import List, Dict, Any

logger = logging.getLogger(__name__)

# ============================================================================
# CATALOG CREATION ORCHESTRATION
# ============================================================================

class CatalogCreationOrchestrator:
    """Orchestrates catalog creation across multiple environments."""
    
    def __init__(self, catalog_mgr: CatalogManager):
        self.catalog_mgr = catalog_mgr
        self.results: List[Dict[str, Any]] = []
    
    def execute(self, environments: List[str], metadata_config: Dict[str, Dict[str, str]]) -> Dict[str, int]:
        """
        Execute catalog creation for all environments.
        
        Args:
            environments: List of environment names
            metadata_config: Catalog metadata configuration
            
        Returns:
            Dict with counts of created and skipped catalogs
        """
        print("="*70)
        print(f"CREATING CATALOGS FOR {PROJECT_NAME} PROJECT")
        print("="*70)
        
        catalogs_created = 0
        catalogs_skipped = 0
        
        for env in environments:
            catalog_name = f"{env}_{PROJECT_NAME}"
            metadata = metadata_config[env]
            
            print(f"\n[{env.upper()} ENVIRONMENT]")
            logger.info(f"Processing environment: {env}")
            
            # Create catalog
            success, message = self.catalog_mgr.create_catalog(
                catalog_name=catalog_name,
                comment=metadata["comment"],
                owner=metadata.get("owner")
            )
            
            # Track results
            self.results.append({
                "environment": env,
                "catalog_name": catalog_name,
                "success": success,
                "message": message
            })
            
            if success:
                catalogs_created += 1
            else:
                catalogs_skipped += 1
        
        # Display summary
        print("\n" + "="*70)
        print(f"SUMMARY: {catalogs_created} catalogs created, {catalogs_skipped} already existed")
        print("="*70)
        
        logger.info(f"Catalog creation complete: {catalogs_created} created, {catalogs_skipped} skipped")
        
        return {
            "created": catalogs_created,
            "skipped": catalogs_skipped,
            "total": len(environments)
        }

# ============================================================================
# EXECUTE CATALOG CREATION
# ============================================================================

orchestrator = CatalogCreationOrchestrator(catalog_manager)
catalog_creation_summary = orchestrator.execute(ENVIRONMENTS, CATALOG_METADATA)

## Step 2: Create Schemas in Each Catalog

Creating the medallion architecture schemas (bronze, silver, gold) plus operational schemas (staging, archive) in each environment.

In [0]:
"""Execute schema creation for all catalogs and environments."""

import logging
from typing import List, Dict, Any

logger = logging.getLogger(__name__)

# ============================================================================
# SCHEMA CREATION ORCHESTRATION
# ============================================================================

class SchemaCreationOrchestrator:
    """Orchestrates schema creation across catalogs and environments."""
    
    def __init__(self, schema_mgr: SchemaManager):
        self.schema_mgr = schema_mgr
        self.results: List[Dict[str, Any]] = []
    
    def execute(
        self,
        environments: List[str],
        data_layers: List[str],
        operational_schemas: List[str],
        schema_metadata: Dict[str, str]
    ) -> Dict[str, int]:
        """
        Execute schema creation for all environments and layers.
        
        Args:
            environments: List of environment names
            data_layers: List of medallion architecture layers
            operational_schemas: List of operational schema names
            schema_metadata: Schema descriptions
            
        Returns:
            Dict with counts of created and skipped schemas
        """
        print("="*70)
        print(f"CREATING SCHEMAS IN {PROJECT_NAME} CATALOGS")
        print("="*70)
        
        schemas_created = 0
        schemas_skipped = 0
        
        # Combine all schemas to create
        all_schemas = data_layers + operational_schemas
        
        for env in environments:
            catalog_name = f"{env}_{PROJECT_NAME}"
            
            print(f"\n[{catalog_name.upper()}]")
            print(f"  Creating {len(all_schemas)} schemas...")
            logger.info(f"Creating schemas for catalog: {catalog_name}")
            
            for schema_name in all_schemas:
                comment = schema_metadata.get(
                    schema_name, 
                    f"{schema_name} schema for {env} environment"
                )
                
                # Create schema
                success, message = self.schema_mgr.create_schema(
                    catalog_name=catalog_name,
                    schema_name=schema_name,
                    comment=comment
                )
                
                # Track results
                self.results.append({
                    "environment": env,
                    "catalog_name": catalog_name,
                    "schema_name": schema_name,
                    "success": success,
                    "message": message
                })
                
                if success:
                    schemas_created += 1
                else:
                    schemas_skipped += 1
        
        # Display summary
        print("\n" + "="*70)
        print(f"SUMMARY: {schemas_created} schemas created, {schemas_skipped} already existed")
        print("="*70)
        
        logger.info(f"Schema creation complete: {schemas_created} created, {schemas_skipped} skipped")
        
        return {
            "created": schemas_created,
            "skipped": schemas_skipped,
            "total": len(environments) * len(all_schemas)
        }

# ============================================================================
# EXECUTE SCHEMA CREATION
# ============================================================================

schema_orchestrator = SchemaCreationOrchestrator(schema_manager)
schema_creation_summary = schema_orchestrator.execute(
    environments=ENVIRONMENTS,
    data_layers=DATA_LAYERS,
    operational_schemas=OPERATIONAL_SCHEMAS,
    schema_metadata=SCHEMA_METADATA
)

## Step 3: Verify Environment Setup

Validating the complete environment structure.

In [0]:
"""Catalog validation and verification module."""

import logging
from typing import List, Dict, Any, Optional
from pyspark.sql import SparkSession

logger = logging.getLogger(__name__)

# ============================================================================
# CATALOG VALIDATOR
# ============================================================================

class CatalogValidator:
    """Validates catalog existence and configuration."""
    
    def __init__(self, spark_session: SparkSession):
        self.spark = spark_session
    
    def get_catalog_info(self, catalog_name: str) -> Optional[Dict[str, str]]:
        """
        Retrieve catalog information.
        
        Args:
            catalog_name: Name of catalog to inspect
            
        Returns:
            Dict with catalog info or None if not found
        """
        try:
            catalog_info = self.spark.sql(
                f"DESCRIBE CATALOG `{catalog_name}`"
            ).collect()
            
            info_dict = {}
            for row in catalog_info:
                info_dict[row.info_name] = row.info_value
            
            return info_dict
            
        except Exception as e:
            logger.error(f"Error retrieving catalog info for {catalog_name}: {e}")
            return None
    
    def validate_all_catalogs(self, environments: List[str], project_name: str) -> Dict[str, Any]:
        """
        Validate all project catalogs.
        
        Args:
            environments: List of environment names
            project_name: Project name
            
        Returns:
            Validation results dictionary
        """
        print("\n" + "="*70)
        print("FINANCE PROJECT - CATALOG VERIFICATION")
        print("="*70 + "\n")
        
        results = {
            "validated": 0,
            "failed": 0,
            "details": []
        }
        
        for env in environments:
            catalog_name = f"{env}_{project_name}"
            
            info = self.get_catalog_info(catalog_name)
            
            if info:
                print(f"✓ {catalog_name}")
                if "Comment" in info:
                    print(f"  └─ {info['Comment']}")
                if "Owner" in info:
                    print(f"  └─ Owner: {info['Owner']}")
                
                results["validated"] += 1
                results["details"].append({
                    "catalog": catalog_name,
                    "status": "exists",
                    "info": info
                })
            else:
                print(f"✗ {catalog_name} - Not found or error")
                results["failed"] += 1
                results["details"].append({
                    "catalog": catalog_name,
                    "status": "not_found"
                })
        
        print("\n" + "="*70)
        logger.info(f"Catalog validation: {results['validated']} validated, {results['failed']} failed")
        
        return results

# ============================================================================
# EXECUTE CATALOG VALIDATION
# ============================================================================

catalog_validator = CatalogValidator(spark)
catalog_validation_results = catalog_validator.validate_all_catalogs(ENVIRONMENTS, PROJECT_NAME)

In [0]:
"""Schema validation and verification module."""

import logging
from typing import List, Dict, Any
from pyspark.sql import SparkSession

logger = logging.getLogger(__name__)

# ============================================================================
# SCHEMA VALIDATOR
# ============================================================================

class SchemaValidator:
    """Validates schema existence and organization."""
    
    def __init__(self, spark_session: SparkSession):
        self.spark = spark_session
    
    def get_schemas_in_catalog(self, catalog_name: str) -> List[str]:
        """
        Get all schemas in a catalog.
        
        Args:
            catalog_name: Name of the catalog
            
        Returns:
            List of schema names
        """
        try:
            schemas = self.spark.sql(
                f"SHOW SCHEMAS IN `{catalog_name}`"
            ).collect()
            return [row.databaseName for row in schemas]
        except Exception as e:
            logger.error(f"Error listing schemas in {catalog_name}: {e}")
            return []
    
    def categorize_schemas(
        self, 
        schemas: List[str], 
        medallion_layers: List[str], 
        operational: List[str]
    ) -> Dict[str, List[str]]:
        """
        Categorize schemas by type.
        
        Args:
            schemas: List of all schema names
            medallion_layers: List of medallion layer names
            operational: List of operational schema names
            
        Returns:
            Dict with categorized schemas
        """
        return {
            "medallion": [s for s in schemas if s in medallion_layers],
            "operational": [s for s in schemas if s in operational],
            "other": [s for s in schemas if s not in medallion_layers and s not in operational]
        }
    
    def validate_all_schemas(
        self,
        environments: List[str],
        project_name: str,
        data_layers: List[str],
        operational_schemas: List[str]
    ) -> Dict[str, Any]:
        """
        Validate schemas across all catalogs.
        
        Args:
            environments: List of environment names
            project_name: Project name
            data_layers: Expected medallion layers
            operational_schemas: Expected operational schemas
            
        Returns:
            Validation results dictionary
        """
        print("\n" + "="*70)
        print("FINANCE PROJECT - SCHEMA VERIFICATION")
        print("="*70 + "\n")
        
        results = {
            "total_schemas": 0,
            "catalogs_validated": 0,
            "details": []
        }
        
        for env in environments:
            catalog_name = f"{env}_{project_name}"
            print(f"\n📁 {catalog_name.upper()}")
            print("   " + "─"*65)
            
            schemas = self.get_schemas_in_catalog(catalog_name)
            
            if not schemas:
                print("   ✗ No schemas found or error accessing catalog")
                continue
            
            categorized = self.categorize_schemas(
                schemas, 
                data_layers, 
                operational_schemas
            )
            
            # Display medallion architecture schemas
            if categorized["medallion"]:
                print("   Medallion Architecture:")
                sorted_medallion = sorted(
                    categorized["medallion"],
                    key=lambda x: data_layers.index(x) if x in data_layers else 999
                )
                for schema in sorted_medallion:
                    print(f"     ✓ {schema}")
            
            # Display operational schemas
            if categorized["operational"]:
                print("   Operational:")
                for schema in categorized["operational"]:
                    print(f"     ✓ {schema}")
            
            # Display other schemas (if any)
            if categorized["other"]:
                print("   Other:")
                for schema in categorized["other"]:
                    print(f"     • {schema}")
            
            print(f"\n   Total: {len(schemas)} schemas")
            
            results["total_schemas"] += len(schemas)
            results["catalogs_validated"] += 1
            results["details"].append({
                "catalog": catalog_name,
                "schema_count": len(schemas),
                "categorized": categorized
            })
        
        print("\n" + "="*70)
        print(f"Total schemas across all catalogs: {results['total_schemas']}")
        print("="*70)
        
        logger.info(f"Schema validation complete: {results['total_schemas']} schemas validated")
        
        return results

# ============================================================================
# EXECUTE SCHEMA VALIDATION
# ============================================================================

schema_validator = SchemaValidator(spark)
schema_validation_results = schema_validator.validate_all_schemas(
    environments=ENVIRONMENTS,
    project_name=PROJECT_NAME,
    data_layers=DATA_LAYERS,
    operational_schemas=OPERATIONAL_SCHEMAS
)

In [0]:
"""Generate comprehensive deployment report."""

import json
from datetime import datetime
from typing import Dict, Any

# ============================================================================
# DEPLOYMENT REPORT GENERATOR
# ============================================================================

class DeploymentReportGenerator:
    """Generates deployment summary and audit reports."""
    
    def __init__(self, project_name: str, version: str):
        self.project_name = project_name
        self.version = version
        self.timestamp = datetime.now()
    
    def generate_summary_report(
        self,
        catalog_summary: Dict[str, int],
        schema_summary: Dict[str, int],
        catalog_validation: Dict[str, Any],
        schema_validation: Dict[str, Any]
    ) -> Dict[str, Any]:
        """
        Generate comprehensive deployment summary.
        
        Args:
            catalog_summary: Catalog creation summary
            schema_summary: Schema creation summary
            catalog_validation: Catalog validation results
            schema_validation: Schema validation results
            
        Returns:
            Complete deployment report
        """
        report = {
            "project_name": self.project_name,
            "version": self.version,
            "deployment_timestamp": self.timestamp.isoformat(),
            "summary": {
                "catalogs": {
                    "created": catalog_summary.get("created", 0),
                    "skipped": catalog_summary.get("skipped", 0),
                    "validated": catalog_validation.get("validated", 0),
                    "total": catalog_summary.get("total", 0)
                },
                "schemas": {
                    "created": schema_summary.get("created", 0),
                    "skipped": schema_summary.get("skipped", 0),
                    "total": schema_validation.get("total_schemas", 0)
                }
            },
            "status": self._determine_deployment_status(
                catalog_validation, 
                schema_validation
            ),
            "validation_details": {
                "catalogs": catalog_validation.get("details", []),
                "schemas": schema_validation.get("details", [])
            }
        }
        
        return report
    
    def _determine_deployment_status(self, catalog_val: Dict, schema_val: Dict) -> str:
        """
        Determine overall deployment status.
        
        Args:
            catalog_val: Catalog validation results
            schema_val: Schema validation results
            
        Returns:
            Status string
        """
        if catalog_val.get("failed", 0) > 0:
            return "FAILED"
        if schema_val.get("total_schemas", 0) == 0:
            return "INCOMPLETE"
        return "SUCCESS"
    
    def display_report(self, report: Dict[str, Any]) -> None:
        """
        Display formatted deployment report.
        
        Args:
            report: Deployment report dictionary
        """
        print("\n" + "="*70)
        print("DEPLOYMENT SUMMARY REPORT")
        print("="*70)
        
        print(f"\nProject: {report['project_name']} v{report['version']}")
        print(f"Deployment Time: {self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Status: {report['status']}")
        
        print("\n" + "-"*70)
        print("RESOURCE CREATION SUMMARY")
        print("-"*70)
        
        cat_summary = report['summary']['catalogs']
        print(f"\nCatalogs:")
        print(f"  ✓ Created: {cat_summary['created']}")
        print(f"  ℹ Skipped (already exist): {cat_summary['skipped']}")
        print(f"  ✓ Validated: {cat_summary['validated']}")
        print(f"  Total: {cat_summary['total']}")
        
        sch_summary = report['summary']['schemas']
        print(f"\nSchemas:")
        print(f"  ✓ Created: {sch_summary['created']}")
        print(f"  ℹ Skipped (already exist): {sch_summary['skipped']}")
        print(f"  Total: {sch_summary['total']}")
        
        print("\n" + "-"*70)
        print("ENVIRONMENT STRUCTURE")
        print("-"*70)
        
        for env_detail in report['validation_details']['schemas']:
            print(f"\n{env_detail['catalog']}:")
            cat_details = env_detail['categorized']
            print(f"  Medallion layers: {len(cat_details['medallion'])}")
            print(f"  Operational schemas: {len(cat_details['operational'])}")
            print(f"  Total: {env_detail['schema_count']}")
        
        print("\n" + "="*70)
        print("DEPLOYMENT COMPLETE")
        print("="*70 + "\n")

# ============================================================================
# GENERATE AND DISPLAY REPORT
# ============================================================================

report_generator = DeploymentReportGenerator(PROJECT_NAME, PROJECT_VERSION)

deployment_report = report_generator.generate_summary_report(
    catalog_summary=catalog_creation_summary,
    schema_summary=schema_creation_summary,
    catalog_validation=catalog_validation_results,
    schema_validation=schema_validation_results
)

report_generator.display_report(deployment_report)

# Log report to file (optional)
logger.info(f"Deployment report: {json.dumps(deployment_report, indent=2)}")

In [0]:
%sql
-- ============================================================================
-- UNITY CATALOG STRUCTURE VERIFICATION
-- ============================================================================

-- List all Finance project catalogs
SHOW CATALOGS LIKE '*_Finance';

-- View specific catalog details
-- DESCRIBE CATALOG EXTENDED Dev_Finance;

-- ============================================================================
-- SCHEMA VERIFICATION
-- ============================================================================

-- Check bronze layer in Development
SHOW TABLES IN Dev_Finance.bronze;

-- Check silver layer in QA
-- SHOW TABLES IN QA_Finance.silver;

-- Check gold layer in Production
-- SHOW TABLES IN Prod_Finance.gold;

-- List all schemas in a catalog
-- SHOW SCHEMAS IN Dev_Finance;

-- ============================================================================
-- DATA FLOW PATHS (MEDALLION ARCHITECTURE)
-- ============================================================================

-- Development Flow:
-- 1. Raw data ingestion    → Dev_Finance.bronze
-- 2. Data cleansing        → Dev_Finance.silver  
-- 3. Business aggregations → Dev_Finance.gold

-- QA Flow (mirrors Dev):
-- 1. Testing raw data      → QA_Finance.bronze
-- 2. Validation cleansed   → QA_Finance.silver
-- 3. Test aggregations     → QA_Finance.gold

-- Production Flow:
-- 1. Production data       → Prod_Finance.bronze
-- 2. Cleansed production   → Prod_Finance.silver
-- 3. Live analytics        → Prod_Finance.gold

In [0]:
"""Example: Create sample tables following enterprise naming conventions.

This cell demonstrates realistic table creation patterns.
Run this only if you want to create sample tables for testing.
"""

from typing import List, Dict
import logging

logger = logging.getLogger(__name__)

# ============================================================================
# TABLE CREATION UTILITIES
# ============================================================================

class TableCreationHelper:
    """Helper class for creating sample tables with standard patterns."""
    
    def __init__(self, spark_session):
        self.spark = spark_session
    
    def create_bronze_table(
        self, 
        catalog: str, 
        table_name: str,
        comment: str = "Bronze layer raw data table"
    ) -> bool:
        """
        Create a bronze layer table for raw data ingestion.
        
        Args:
            catalog: Catalog name (e.g., 'Dev_Finance')
            table_name: Table name (e.g., 'transaction_logs')
            comment: Table description
            
        Returns:
            bool: Success status
        """
        try:
            ddl = f"""
            CREATE TABLE IF NOT EXISTS `{catalog}`.bronze.`{table_name}` (
                record_id STRING COMMENT 'Unique record identifier',
                source_system STRING COMMENT 'Source system name',
                ingestion_timestamp TIMESTAMP COMMENT 'Data ingestion time',
                raw_data STRING COMMENT 'Raw JSON payload from source',
                ingestion_date DATE COMMENT 'Partition key - ingestion date'
            )
            USING DELTA
            COMMENT '{comment}'
            PARTITIONED BY (ingestion_date)
            TBLPROPERTIES (
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true',
                'quality' = 'bronze'
            )
            """
            
            self.spark.sql(ddl)
            logger.info(f"Created bronze table: {catalog}.bronze.{table_name}")
            print(f"  ✓ Created: {catalog}.bronze.{table_name}")
            return True
            
        except Exception as e:
            logger.error(f"Error creating bronze table: {e}")
            print(f"  ✗ Error: {str(e)}")
            return False
    
    def create_silver_table(
        self,
        catalog: str,
        table_name: str,
        comment: str = "Silver layer cleansed data table"
    ) -> bool:
        """
        Create a silver layer table for cleansed data.
        
        Args:
            catalog: Catalog name
            table_name: Table name
            comment: Table description
            
        Returns:
            bool: Success status
        """
        try:
            ddl = f"""
            CREATE TABLE IF NOT EXISTS `{catalog}`.silver.`{table_name}` (
                transaction_id STRING COMMENT 'Business transaction ID',
                transaction_date DATE COMMENT 'Transaction date',
                account_number STRING COMMENT 'Customer account number',
                transaction_amount DECIMAL(18,2) COMMENT 'Transaction amount',
                currency_code STRING COMMENT 'ISO currency code',
                transaction_type STRING COMMENT 'Transaction type',
                status STRING COMMENT 'Transaction status',
                created_timestamp TIMESTAMP COMMENT 'Record creation time',
                modified_timestamp TIMESTAMP COMMENT 'Last modification time',
                data_quality_score DOUBLE COMMENT 'DQ score (0-1)',
                processing_date DATE COMMENT 'Partition key'
            )
            USING DELTA
            COMMENT '{comment}'
            PARTITIONED BY (processing_date)
            TBLPROPERTIES (
                'delta.enableChangeDataFeed' = 'true',
                'quality' = 'silver'
            )
            """
            
            self.spark.sql(ddl)
            logger.info(f"Created silver table: {catalog}.silver.{table_name}")
            print(f"  ✓ Created: {catalog}.silver.{table_name}")
            return True
            
        except Exception as e:
            logger.error(f"Error creating silver table: {e}")
            print(f"  ✗ Error: {str(e)}")
            return False
    
    def create_gold_table(
        self,
        catalog: str,
        table_name: str,
        comment: str = "Gold layer aggregated metrics table"
    ) -> bool:
        """
        Create a gold layer table for business metrics.
        
        Args:
            catalog: Catalog name
            table_name: Table name
            comment: Table description
            
        Returns:
            bool: Success status
        """
        try:
            ddl = f"""
            CREATE TABLE IF NOT EXISTS `{catalog}`.gold.`{table_name}` (
                report_date DATE COMMENT 'Reporting date',
                account_segment STRING COMMENT 'Account segment',
                total_transactions BIGINT COMMENT 'Total transaction count',
                total_amount DECIMAL(18,2) COMMENT 'Total transaction amount',
                avg_transaction_amount DECIMAL(18,2) COMMENT 'Average transaction amount',
                unique_accounts INT COMMENT 'Unique account count',
                created_timestamp TIMESTAMP COMMENT 'Record creation time'
            )
            USING DELTA
            COMMENT '{comment}'
            PARTITIONED BY (report_date)
            TBLPROPERTIES (
                'quality' = 'gold',
                'business_unit' = 'finance'
            )
            """
            
            self.spark.sql(ddl)
            logger.info(f"Created gold table: {catalog}.gold.{table_name}")
            print(f"  ✓ Created: {catalog}.gold.{table_name}")
            return True
            
        except Exception as e:
            logger.error(f"Error creating gold table: {e}")
            print(f"  ✗ Error: {str(e)}")
            return False

# ============================================================================
# EXAMPLE USAGE (COMMENTED OUT - UNCOMMENT TO RUN)
# ============================================================================

# Uncomment the following lines to create sample tables

# table_helper = TableCreationHelper(spark)

# print("\nCreating sample tables in Dev_Finance...")
# table_helper.create_bronze_table("Dev_Finance", "raw_transactions", "Raw transaction logs from source systems")
# table_helper.create_silver_table("Dev_Finance", "transactions_cleansed", "Cleansed and validated transactions")
# table_helper.create_gold_table("Dev_Finance", "daily_transaction_summary", "Daily transaction metrics by segment")

print("\nℹ Example code ready. Uncomment lines above to create sample tables.")

## ✅ Deployment Complete - Implementation Guide

### Environment Structure Created

**Unity Catalog Hierarchy:**
```
Workspace
└── Dev_Finance (Development Catalog)
    ├── bronze   (Raw data ingestion)
    ├── silver   (Cleansed data)
    ├── gold     (Business metrics)
    ├── staging  (Temporary processing)
    └── archive  (Historical retention)
└── QA_Finance (QA Catalog)
    ├── bronze, silver, gold, staging, archive
└── Prod_Finance (Production Catalog)
    └── bronze, silver, gold, staging, archive
```

**Total Resources:** 3 catalogs × 5 schemas = 15 schemas

---

### Enterprise Development Workflow

#### 1. Development Environment (Dev_Finance)
- **Purpose**: Active development and experimentation
- **Data Retention**: 30 days
- **Access**: Data engineering team
- **Activities**:
  * Build and test data pipelines
  * Experiment with transformations
  * Initial data quality checks
  * Unit testing

#### 2. QA Environment (QA_Finance)
- **Purpose**: Validation and integration testing
- **Data Retention**: 90 days
- **Access**: QA team
- **Activities**:
  * Integration testing
  * Data validation
  * Performance testing
  * UAT preparation

#### 3. Production Environment (Prod_Finance)
- **Purpose**: Live business operations
- **Data Retention**: 7 years (compliance)
- **Access**: Data platform team (restricted)
- **Activities**:
  * Production data processing
  * Business analytics
  * Reporting and dashboards
  * Strict change control

---

### Medallion Architecture Best Practices

#### Bronze Layer (Raw Data Zone)
**Naming Convention:** `{env}_Finance.bronze.{source_system}_{entity}_raw`

**Purpose:**
* Land raw data exactly as received
* Append-only pattern (immutable)
* Full audit trail

**Example:**
```python
# Bronze table for transaction logs
Dev_Finance.bronze.erp_transactions_raw
Dev_Finance.bronze.crm_customer_raw
```

**Table Properties:**
* Partitioned by ingestion_date
* Enable Change Data Feed
* Store source metadata

#### Silver Layer (Cleansed Data Zone)
**Naming Convention:** `{env}_Finance.silver.{entity}_cleansed`

**Purpose:**
* Apply business rules
* Data quality validation
* Type conversions
* Deduplication

**Example:**
```python
# Silver table for validated transactions
Dev_Finance.silver.transactions_cleansed
Dev_Finance.silver.customers_validated
```

**Table Properties:**
* Partitioned by processing_date
* Include data quality scores
* Add business key constraints

#### Gold Layer (Business Metrics Zone)
**Naming Convention:** `{env}_Finance.gold.{metric}_{aggregation}_{grain}`

**Purpose:**
* Business-level aggregations
* KPIs and metrics
* Optimized for BI consumption

**Example:**
```python
# Gold table for daily summaries
Dev_Finance.gold.revenue_summary_daily
Dev_Finance.gold.customer_metrics_monthly
```

**Table Properties:**
* Highly optimized (Z-ORDER)
* Denormalized for performance
* Business glossary tags

---

### Code Standards & Naming Conventions

#### Table Naming
* Use lowercase with underscores
* Include layer prefix: `bronze_`, `silver_`, `gold_`
* Format: `{layer}_{domain}_{entity}_{qualifier}`
* Examples:
  - `bronze_erp_invoices_raw`
  - `silver_customer_accounts_validated`
  - `gold_sales_metrics_daily`

#### Column Naming
* Descriptive lowercase with underscores
* Include units in name: `amount_usd`, `duration_seconds`
* Timestamp fields: `created_timestamp`, `modified_timestamp`
* Date partitions: `ingestion_date`, `processing_date`, `report_date`

#### Code Structure
* Use classes for related functionality
* Type hints on all function parameters
* Comprehensive docstrings (Google style)
* Logging at INFO level for operations
* Exception handling with context

---

### Next Steps

1. **Configure Access Controls**
   ```sql
   -- Grant appropriate permissions
   GRANT USE CATALOG ON CATALOG Dev_Finance TO `data_engineering_team`;
   GRANT CREATE SCHEMA ON CATALOG Dev_Finance TO `data_engineering_team`;
   ```

2. **Set Up Data Pipelines**
   * Configure Auto Loader for bronze ingestion
   * Build silver transformation pipelines
   * Create gold aggregation jobs

3. **Implement Data Quality**
   * Add expectations in Delta Live Tables
   * Set up data quality monitoring
   * Configure alerting

4. **Documentation**
   * Document table schemas in Unity Catalog
   * Maintain data lineage
   * Update operational runbooks

---

### Security & Governance

🔒 **Access Control Checklist:**
- [ ] Configure catalog-level permissions
- [ ] Set schema-level access controls
- [ ] Enable audit logging
- [ ] Configure row/column-level security
- [ ] Set up data classification tags
- [ ] Configure encryption at rest
- [ ] Enable secure views for PII

---

### Support & Resources

* **Architecture Documentation**: Enterprise Data Platform Wiki
* **Change Management**: Submit via JIRA (DATA project)
* **Support**: data-platform-team@company.com
* **On-Call**: Refer to PagerDuty rotation